# Rotax 914 ULF - Train the model

One network answers six questions at once - is something wrong, which fault,
how bad, which sensor is lying and how, and how worn the engine is. Trained jointly
so no answer is forgotten while another is learned (v4's weakness).

Command-line equivalent: `validation/venv/bin/python validation_v5/run_v5.py 914`

In [ ]:
# Finds validation_v5 wherever Jupyter was started, then loads the helpers.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers_v5.py')): break
    if os.path.exists(os.path.join(d, 'validation_v5', 'nb_helpers_v5.py')):
        d = os.path.join(d, 'validation_v5'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers_v5 import *

ENGINE = '914'
setup(ENGINE)

### 1. Train (or load)
40 passes at most, stopping when the validation score stops improving. The kept
version is the one with the best validation composite, not the lowest loss.
Runs on the Metal GPU.

In [ ]:
ensure(ENGINE, 'train')      # ensure(ENGINE, 'train', force=True) to retrain

### 2. How training went

In [ ]:
plot_history(ENGINE)

### What this means
Scores that rise and flatten mean the model learned and stopped before
memorising; the dashed line is the version kept. These are VALIDATION flights -
the honest score comes only from the test notebook.

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - what the model looks at for one answer. |
| **Healthy twin** | A perfectly healthy simulated copy of the engine flown through the same flight. |
| **Residual** | Reading minus the healthy twin's reading, in units of the sensor's noise (sigma). |
| **Train / cal / val / test** | Flights used to learn / to calibrate / to choose / to score. A flight is in one only. |
| **Baseline** | A do-nothing rule the model must beat by a clear margin, or its gate fails. |
| **95% interval** | The range the score would take on other test flights like these (flight bootstrap). |
| **TBO** | Time between overhauls: 2000 h (1200 h for the 915). |